In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [7]:
df["Class"].nunique()

7

### X Y Split

In [8]:
X = df.drop(columns = ["Class"])
y = df["Class"]

### Label Encoding

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

### Train Test Split

In [14]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42 
)

### Scaling

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Deep Learning

### Tensors

In [17]:
import torch
import torch.nn as nn

In [19]:
X_train_tensor = torch.tensor(X_train_scaled , dtype = torch.float32)
y_train_tensor = torch.tensor(y_train , dtype = torch.long)

X_test_tensor = torch.tensor(X_test_scaled , dtype = torch.float32)
y_test_tensor = torch.tensor(y_test , dtype = torch.long)

### TensorDataset and DataLoader

In [21]:
from torch.utils.data import TensorDataset , DataLoader


train_dataset = TensorDataset(X_train_tensor , y_train_tensor)
test_dataset = TensorDataset(X_test_tensor , y_test_tensor)

train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle = True)

### Defining ANN model

In [28]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer 
            nn.Linear(X.shape[1],64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64,64),
            nn.ReLU(),

            # Output layer
            nn.Linear(64 , 7),
        )

    def forward(self,x):
        return self.model(x)

In [29]:
model = ANN()

### Loss Function and Optimizer

In [30]:
import torch.optim as optim

crietrion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the ANN model

In [31]:
epochs = 100 

for epoch in  range(epochs):
    model.train()

    running_loss = 0.0

    for xb , yb in train_loader:

        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = crietrion(outputs,yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()


    avg_training_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}  training loss = {avg_training_loss}")
    

Epoch 1  training loss = 1.7126293700674307
Epoch 2  training loss = 1.104562909706779
Epoch 3  training loss = 0.7035179786060167
Epoch 4  training loss = 0.5311819509319637
Epoch 5  training loss = 0.45102050252582715
Epoch 6  training loss = 0.3905805537234182
Epoch 7  training loss = 0.3516420922849489
Epoch 8  training loss = 0.3052011780116869
Epoch 9  training loss = 0.2818116474410762
Epoch 10  training loss = 0.2666285575731941
Epoch 11  training loss = 0.24367195173450137
Epoch 12  training loss = 0.23047468001427857
Epoch 13  training loss = 0.2171433775321297
Epoch 14  training loss = 0.20246129126652426
Epoch 15  training loss = 0.19975227700627368
Epoch 16  training loss = 0.1872901135812635
Epoch 17  training loss = 0.18244231929597649
Epoch 18  training loss = 0.17380993780882462
Epoch 19  training loss = 0.165855020608591
Epoch 20  training loss = 0.16446163748269496
Epoch 21  training loss = 0.1603057758639688
Epoch 22  training loss = 0.1507418685309265
Epoch 23  tra

### Evaluate

In [32]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb , yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs,1) # it return max value and its index

        correct += (predicted == yb).sum().item() # correct samples in each batch
        total += yb.size(0) # actual samples in each batch


print(f"Accuracy : {(correct / total)*100} ")

Accuracy : 95.55555555555556 
